# Кластеризация: как найти группы без готовых ответов

Сегодня попробуем три способа искать группы: **KMeans**, **агломеративную кластеризацию** и **DBSCAN**. Наша цель - понять их идеи и научиться осмысленно читать результат. Запускать ячейки нужно сверху вниз.

Нужны `pandas`, `matplotlib`, `scikit-learn` и Jupyter. Набор Iris входит в `scikit-learn` и не требует скачивания. Источник: [документация load_iris](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html).

## 1. Подготовка

Импорты и функцию для рисунков оставляем готовыми: сегодня нас интересуют прежде всего данные и поведение алгоритмов. Функция `draw_clusters` показывает точки и подписывает цвета номером кластера. Номер - условное имя, а не оценка качества.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score

plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True


def draw_clusters(points, labels, title, centers=None):
    labels = np.asarray(labels)
    fig, ax = plt.subplots()
    for label in np.unique(labels):
        mask = labels == label
        name = "шум (-1)" if label == -1 else f"кластер {label}"
        ax.scatter(points[mask, 0], points[mask, 1], label=name, alpha=0.8, s=40)
    if centers is not None:
        ax.scatter(centers[:, 0], centers[:, 1], marker="X", s=180,
                   c="black", label="центры")
    ax.set(xlabel="Длина лепестка (стандарт.)",
           ylabel="Ширина лепестка (стандарт.)", title=title)
    ax.legend()
    plt.show()

## 2. Знакомство с данными

Загрузите Iris. Посмотрите названия столбцов и первые пять строк. Затем оставьте только **длину и ширину лепестка** - два признака удобно показать на плоскости. Имена этих столбцов можно взять из списка `iris.feature_names[2:4]`.

Вопрос перед запуском: почему число вида (`target`) пока не должно попадать в `X`?

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()
print(df.shape)
print(df.columns.tolist())
display(...)  # первые пять строк

feature_names = iris.feature_names[2:4]
X = ...  # датафрейм из двух столбцов
print(X.describe())

Постройте диаграмму рассеяния двух выбранных признаков. Пока все точки одного цвета: меток групп у нас нет. Есть ли на глаз отдельная группа? Сколько ещё групп вы бы предположили?

In [ ]:
fig, ax = plt.subplots()
ax.scatter(..., ..., alpha=0.8)
ax.set(xlabel=feature_names[0], ylabel=feature_names[1],
       title="Iris: два признака до кластеризации")
plt.show()

Алгоритмы ниже работают с расстояниями. Поэтому приведём оба столбца к общему масштабу: после `StandardScaler` среднее каждого столбца около нуля, стандартное отклонение около единицы. Это не добавляет информации, но меняет единицы измерения. Сохраните результат в `X_scaled`; дальше все методы получат именно его.

In [ ]:
scaler = StandardScaler()
X_scaled = ...  # обучить scaler на X и преобразовать X
print(np.round(X_scaled.mean(axis=0), 2))
print(np.round(X_scaled.std(axis=0), 2))

## 3. KMeans

Идея: заранее говорим, сколько групп хотим (`k`). Метод размещает `k` центров, относит каждую точку к ближайшему, пересчитывает центры и повторяет. Он **обязательно** назначает группу каждой точке. Число `random_state` делает результат повторяемым.

Создайте модель с тремя кластерами, получите метки и нарисуйте их вместе с центрами. Переменные ниже будут нужны и в следующих заданиях.

In [ ]:
kmeans = KMeans(n_clusters=..., random_state=42, n_init=10)
labels_kmeans = ...  # fit_predict на X_scaled
print(pd.Series(labels_kmeans).value_counts().sort_index())
draw_clusters(X_scaled, labels_kmeans, "KMeans, k=3",
              centers=...)  # центры модели

Проверим, как выбор `k` влияет на результат. Для `k = 2, 3, 4` вычислите **силуэт**: значение ближе к 1 означает, что точки в среднем ближе к своей группе и дальше от соседней; около 0 - группы соприкасаются. Это подсказка, а не автоматический ответ. Силуэт определён только при числе групп от 2 до `n_samples - 1`.

Запишите: какое `k` получило наибольший силуэт? Значит ли это, что оно единственно правильное?

In [ ]:
for k in [2, 3, 4]:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = ...
    score = ...
    print(f"k={k}: силуэт={score:.3f}")

## 4. Агломеративная кластеризация

Здесь сначала каждая точка - отдельная группа. Затем ближайшие группы последовательно объединяются. Вариант `ward` старается не допустить большого разброса внутри новых групп. Зададим три итоговых кластера, чтобы сравнить с KMeans.

Получите метки, размеры групп и рисунок. Похожи ли границы на результат KMeans? Номера групп сравнивать напрямую нельзя: они произвольны.

In [ ]:
agglomerative = AgglomerativeClustering(n_clusters=..., linkage="ward")
labels_agg = ...
print(pd.Series(labels_agg).value_counts().sort_index())
draw_clusters(X_scaled, labels_agg, "Агломеративный метод, 3 группы")

## 5. DBSCAN

DBSCAN ищет плотные области. `eps` - радиус соседства **в стандартизированных единицах**, `min_samples` - минимальное число точек в этом соседстве, включая саму точку. Метка `-1` означает шум: точка не попала ни в один найденный кластер. Число групп заранее не задаём.

Начните с `eps=0.25` и `min_samples=5`. Посчитайте группы и шум. Здесь ожидаются две группы и несколько точек с меткой `-1`: это не ошибка кода. Сравните рисунок с предыдущими.

In [ ]:
dbscan = DBSCAN(eps=..., min_samples=...)
labels_dbscan = ...
print(pd.Series(labels_dbscan).value_counts().sort_index())
draw_clusters(X_scaled, labels_dbscan, "DBSCAN, eps=0.25")

Измените только `eps` на `0.35`. Сколько стало точек шума? Почему метод мог объединить группы или присоединить точки? Не подбирайте параметр по названиям видов: мы ещё не открывали ответы.

In [ ]:
dbscan_wide = DBSCAN(eps=..., min_samples=5)
labels_dbscan_wide = ...
print(pd.Series(labels_dbscan_wide).value_counts().sort_index())
draw_clusters(X_scaled, labels_dbscan_wide, "DBSCAN, eps=0.35")

## 6. Сравнение

Составьте небольшую таблицу: метод, число найденных групп **без шума**, число шумовых точек. Для силуэта DBSCAN будем считать только точки, которым присвоена группа. Поэтому его оценка относится к меньшему набору строк и не является прямым соревнованием с остальными двумя значениями.

In [ ]:
def summarize(name, labels, points):
    labels = np.asarray(labels)
    assigned = labels != -1
    n_clusters = ...  # уникальные метки без -1
    n_noise = ...
    score = ...  # силуэт только для assigned, если групп хотя бы две
    return {"Метод": name, "Групп": n_clusters,
            "Шумовых точек": n_noise, "Силуэт*": score}


results = pd.DataFrame([
    summarize("KMeans", labels_kmeans, X_scaled),
    summarize("Агломеративный", labels_agg, X_scaled),
    summarize("DBSCAN (0.25)", labels_dbscan, X_scaled),
])
display(results)

Только теперь посмотрим на известные виды Iris. Сделайте таблицу частот: строки - кластеры KMeans, столбцы - вид. Это не часть обучения модели. Кластер 0 не обязан означать вид 0; и биологические виды не обязаны выглядеть как три компактных облака на двух выбранных признаках.

In [ ]:
comparison = pd.crosstab(
    pd.Series(labels_kmeans, name="Кластер KMeans"),
    pd.Series(iris.target_names[iris.target], name="Вид"),
)
display(...)  # comparison

## 7. Итог

Запишите 3–4 предложения своими словами:

1. Что у трёх методов общее?
2. Какую настройку нужно задать у KMeans и агломеративного метода, а какую - у DBSCAN?
3. Что означает метка `-1`? Почему силуэт сам по себе не доказывает, что группы «правильные»?
4. Для примера из транспорта - группировки остановок по пассажиропотоку и времени ожидания - какие дополнительные данные и проверки потребуются до управленческого решения?

**По желанию:** повторите опыт с двумя признаками чашелистика (`iris.feature_names[:2]`) и сравните рисунки. Не ожидайте тех же результатов: расстояния между точками изменятся.

Справка: [обзор методов кластеризации scikit-learn](https://scikit-learn.org/stable/modules/clustering.html), [silhouette_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).